In [0]:
%run ../../02_common_utils/operations

In [0]:
from datetime import datetime
from pyspark.sql.functions import col, lit, current_timestamp, date_format, concat, expr, lead, when, md5, unix_date, max, min, trim, abs as spark_abs, hash as spark_hash
from pyspark.sql.window import Window
from pyspark.sql.types import LongType, DecimalType, IntegerType

team_name = "team_lemma"
silver_db = f"charles_schwab_retailbrokerage_dev_{team_name}.silver"
staging_db = f"charles_schwab_retailbrokerage_dev_{team_name}.staging"
gold_db = f"charles_schwab_retailbrokerage_dev_{team_name}.gold"
# Extract the carried run_id AND batch from the upstream staging table
try:
    run_info_row = spark.sql(f"SELECT _run_id, _batch FROM {staging_db}.finwire_parsed LIMIT 1").first()
    carried_run_id = run_info_row[0] if run_info_row else "unknown"
    carried_batch = run_info_row[1] if run_info_row else "unknown"
except Exception:
    carried_run_id = "unknown"
    carried_batch = "unknown"
run_id=carried_run_id
spark.sql(f"USE CATALOG charles_schwab_retailbrokerage_dev_{team_name}")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

In [0]:
log_pipeline_message(spark, carried_run_id, 'INFO', 'gold_market_all', 'Starting processing for gold market tables')
start_pipeline_run(spark, carried_run_id, carried_batch)
log_domain_run_status(spark, carried_run_id, carried_batch, 'MARKET', 'RUNNING')

In [0]:
cmp_stg = spark.table(f"{staging_db}.finwire_parsed").filter(col("RecType") == "CMP")

w_scd = Window.partitionBy("CompanyID").orderBy("EffectiveDate")

dim_company = (
    cmp_stg
    .withColumn("enddate", lead("EffectiveDate").over(w_scd))
    .withColumn("enddate", when(col("enddate").isNull(), expr("CAST('9999-12-31' AS DATE)")).otherwise(col("enddate")))
    .withColumn("iscurrent", col("enddate") == expr("CAST('9999-12-31' AS DATE)"))
    .withColumn("valid_from", col("EffectiveDate"))
    .withColumn("valid_to", col("enddate"))
    .withColumn("islowgrade", col("SPrating") < lit("BBB"))
    .withColumn("sk_companyid", (concat(date_format(col("EffectiveDate"), "yyyyMMdd"), col("CompanyID"))).cast(LongType()))
    .withColumn("version_number", expr("row_number() over (partition by CompanyID order by EffectiveDate)"))
    .withColumn("record_hash", md5(concat(col("CompanyName"), col("IndustryID"), col("Status"), col("SPrating"), col("CEOname"))))
    .withColumn("system_valid_from", current_timestamp())
    .withColumn("system_valid_to", expr("CAST('9999-12-31 23:59:59' AS TIMESTAMP)"))
    .select(
        col("sk_companyid"),
        col("CompanyID").alias("companyid"),
        col("Status").alias("status"),
        col("CompanyName").alias("name"),
        col("IndustryID").alias("industry"),
        col("SPrating").alias("sprating"),
        col("islowgrade"),
        col("CEOname").alias("ceo"),
        col("AddrLine1").alias("addressline1"),
        col("AddrLine2").alias("addressline2"),
        col("PostalCode").alias("postalcode"),
        col("City").alias("city"),
        col("StateProvince").alias("stateprov"),
        col("Country").alias("country"),
        col("Description").alias("description"),
        col("FoundingDate").alias("foundingdate"),
        col("iscurrent"),
        col("valid_from"),
        col("valid_to"),
        col("EffectiveDate").alias("effectivedate"),
        col("enddate"),
        col("version_number"),
        col("record_hash"),
        col("system_valid_from"),
        col("system_valid_to"),
        col("_batch"),
        lit(run_id).alias("_run_id"),
        current_timestamp().alias("_load_ts")
    )
)

(dim_company.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{gold_db}.dim_company"))

In [0]:
from pyspark.sql.functions import greatest, least

# 1. Read SEC records from staging
sec_stg = spark.table(f"{staging_db}.finwire_parsed").filter(col("RecType").isin("SEC_CIK", "SEC_NAME"))

# 2. Extract First Trade Dates from Silver Market History
dm = spark.table(f"{silver_db}.markethistory")
first_trade = dm.groupBy("dm_s_symb").agg(min("dm_date").alias("ft_first_trade_date"))

# 3. Resolve CompanyID
company_lookup = spark.table(f"{silver_db}.company").select("companyname", col("companyid").alias("lkp_companyid")).dropDuplicates(["companyname"])

sec_resolved = (
    sec_stg
    .join(company_lookup, sec_stg.CoNameOrCIK == company_lookup.companyname, "left")
    .withColumn("resolved_companyid", 
                when(col("RecType") == "SEC_CIK", col("CoNameOrCIK").cast(LongType()))
                .otherwise(col("lkp_companyid")))
)

# 4. Calculate initial ValidFrom and ValidTo for the Security BEFORE the join
w_sec = Window.partitionBy("Symbol").orderBy("EffectiveDate")
sec_dates = (
    sec_resolved
    .withColumn("sec_valid_from", col("EffectiveDate"))
    .withColumn("sec_valid_to", lead(col("EffectiveDate")).over(w_sec))
    .withColumn("sec_valid_to", when(col("sec_valid_to").isNull(), expr("CAST('9999-12-31' AS DATE)")).otherwise(col("sec_valid_to")))
)

# 5. Fetch dim_company with aliased columns
dim_comp = spark.table(f"{gold_db}.dim_company").select(
    col("sk_companyid"), 
    col("companyid").alias("dim_companyid"), 
    col("effectivedate").alias("comp_effectivedate"), 
    col("enddate").alias("comp_enddate")
)

# 6. OVERLAP JOIN (Interval Expansion)
# This splits the security row if the company has multiple SCD-2 records during the security's lifespan!
sec_expanded = (
    sec_dates
    .join(
        dim_comp,
        (sec_dates.resolved_companyid == dim_comp.dim_companyid) &
        (sec_dates.sec_valid_from < dim_comp.comp_enddate) &
        (sec_dates.sec_valid_to > dim_comp.comp_effectivedate),
        "left"
    )
)

# 7. Calculate new fragmented Effective/End Dates for the split intervals
sec_final_dates = (
    sec_expanded
    .withColumn("expanded_effective_date", 
                when(col("comp_effectivedate").isNull(), col("sec_valid_from"))
                .otherwise(greatest(col("sec_valid_from"), col("comp_effectivedate"))))
    .withColumn("expanded_end_date", 
                when(col("comp_enddate").isNull(), col("sec_valid_to"))
                .otherwise(least(col("sec_valid_to"), col("comp_enddate"))))
)

# 8. Join First Trade Date
sec_with_ft = sec_final_dates.join(first_trade, sec_final_dates.Symbol == first_trade.dm_s_symb, "left")

# 9. Apply Final SCD-2 Logic & Build Surrogate Keys
dim_security = (
    sec_with_ft
    .withColumn("iscurrent", col("expanded_end_date") == expr("CAST('9999-12-31' AS DATE)"))
    .withColumn("valid_from", col("expanded_effective_date"))
    .withColumn("valid_to", col("expanded_end_date"))
    # SK Formula based on the NEW expanded effective date
    .withColumn("sk_securityid", concat(date_format(col("expanded_effective_date"), "yyyyMMdd"), spark_abs(spark_hash(col("Symbol")))).cast(LongType()))
    .withColumn("version_number", expr("row_number() over (partition by Symbol order by expanded_effective_date)"))
    .withColumn("record_hash", md5(concat(col("Name"), col("ExID"), col("Status"))))
    .withColumn("system_valid_from", current_timestamp())
    .withColumn("system_valid_to", expr("CAST('9999-12-31 23:59:59' AS TIMESTAMP)"))
    .select(
        col("sk_securityid"),
        col("Symbol").alias("symbol"),
        col("IssueType").alias("issue"),
        col("Status").alias("status"),
        col("Name").alias("name"),
        col("ExID").alias("exchangeid"),
        col("sk_companyid"),
        col("CoNameOrCIK").alias("conameorcik"),
        col("ShOut").alias("sharesoutstanding"),
        col("ft_first_trade_date").alias("firsttrade"),
        col("ft_first_trade_date").alias("firsttradeonexchange"),
        col("Dividend").alias("dividend"),
        col("iscurrent"),
        col("valid_from"),
        col("valid_to"),
        col("expanded_effective_date").alias("effectivedate"),
        col("expanded_end_date").alias("enddate"),
        col("version_number"),
        col("record_hash"),
        col("system_valid_from"),
        col("system_valid_to"),
        col("_batch"),
        lit(run_id).alias("_run_id"),
        current_timestamp().alias("_load_ts")
    )
)

# Write to Gold
(dim_security.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{gold_db}.dim_security"))

In [0]:
fin_stg = spark.table(f"{staging_db}.finwire_parsed").filter(col("RecType").isin("FIN_COMPANYID", "FIN_NAME"))

fin_resolved = (
    fin_stg
    .join(company_lookup, fin_stg.CoNameOrCIK == company_lookup.companyname, "left")
    .withColumn("resolved_companyid", 
                when(col("RecType") == "FIN_COMPANYID", col("CoNameOrCIK").cast(LongType()))
                .otherwise(col("lkp_companyid")))
)

# Use the newly aliased comp_effectivedate and comp_enddate
fin_with_comp = (
    fin_resolved
    .join(
        dim_comp,
        (fin_resolved.resolved_companyid == dim_comp.dim_companyid) &
        (fin_resolved.PostingDate >= dim_comp.comp_effectivedate) &
        (fin_resolved.PostingDate < dim_comp.comp_enddate),
        "left"
    )
)

financial = (
    fin_with_comp
    .select(
        col("sk_companyid"),
        col("FIYear").cast(IntegerType()).alias("fi_year"),
        col("FIQtr").cast(IntegerType()).alias("fi_qtr"),
        col("QtrStartDate").alias("fi_qtr_start_date"),
        col("Revenue").cast(DecimalType(15,2)).alias("fi_revenue"),
        col("Earnings").cast(DecimalType(15,2)).alias("fi_net_earn"),
        col("EPS").cast(DecimalType(10,2)).alias("fi_basic_eps"),
        col("DilutedEPS").cast(DecimalType(10,2)).alias("fi_dilut_eps"),
        col("Margin").cast(DecimalType(10,2)).alias("fi_margin"),
        col("Inventory").cast(DecimalType(15,2)).alias("fi_inventory"),
        col("Assets").cast(DecimalType(15,2)).alias("fi_assets"),
        col("Liabilities").cast(DecimalType(15,2)).alias("fi_liability"),
        col("ShOut").cast(LongType()).alias("fi_out_basic"),
        col("DilutedShOut").cast(LongType()).alias("fi_out_dilut"),
        col("_batch"),
        lit(run_id).alias("_run_id"),
        current_timestamp().alias("_load_ts")
    )
)

(financial.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{gold_db}.financial"))

In [0]:
from pyspark.sql.functions import col, lit, current_timestamp, date_format, expr, lead, when, max, min, unix_date
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, DecimalType

dm_sil = spark.table(f"{silver_db}.markethistory")
dim_sec = spark.table(f"{gold_db}.dim_security").select("symbol", "effectivedate", "enddate", "sk_securityid", "sk_companyid", "dividend")

# FIX: Alias sk_companyid to fin_sk_companyid so it doesn't clash during the join
fin_gold = spark.table(f"{gold_db}.financial").select(col("sk_companyid").alias("fin_sk_companyid"), "fi_qtr_start_date", "fi_basic_eps")

dm_with_sec = dm_sil.join(
    dim_sec,
    (dm_sil.dm_s_symb == dim_sec.symbol) &
    (dm_sil.dm_date >= dim_sec.effectivedate) &
    (dm_sil.dm_date < dim_sec.enddate),
    "left"
)

# Use the new aliased name for the Window partition
w_fin = Window.partitionBy("fin_sk_companyid").orderBy("fi_qtr_start_date")

fin_gold_optimized = (
    fin_gold
    .withColumn("fi_qtr_end_date", lead("fi_qtr_start_date").over(w_fin))
    .withColumn("fi_qtr_end_date", when(col("fi_qtr_end_date").isNull(), expr("CAST('9999-12-31' AS DATE)")).otherwise(col("fi_qtr_end_date")))
)

# Join using the distinct column names
dm_fin_join = dm_with_sec.join(
    fin_gold_optimized,
    (dm_with_sec.sk_companyid == fin_gold_optimized.fin_sk_companyid) &
    (dm_with_sec.dm_date >= fin_gold_optimized.fi_qtr_start_date) &
    (dm_with_sec.dm_date < fin_gold_optimized.fi_qtr_end_date),
    "left"
)

w_52 = Window.partitionBy("sk_securityid").orderBy(unix_date("dm_date")).rangeBetween(-364, 0)

fact_markethistory = (
    dm_fin_join
    .withColumn("sk_dateid", date_format("dm_date", "yyyyMMdd").cast(IntegerType()))
    .withColumn("peratio", expr("try_divide(dm_close, fi_basic_eps)").cast(DecimalType(10,2)))
    .withColumn("yield", expr("try_divide(dividend, dm_close)").cast(DecimalType(10,6)))
    .withColumn("fiftytwoweekhigh", max("dm_high").over(w_52).cast(DecimalType(8,2)))
    .withColumn("fiftytwoweeklow", min("dm_low").over(w_52).cast(DecimalType(8,2)))
    .select(
        col("sk_securityid"),
        col("sk_companyid"), # Now unambiguous!
        col("sk_dateid"),
        col("peratio"),
        col("yield"),
        col("fiftytwoweekhigh"),
        col("fiftytwoweeklow"),
        col("dm_close").alias("closeprice"),
        col("dm_high").alias("dayhigh"),
        col("dm_low").alias("daylow"),
        col("dm_vol").alias("volume"),
        dm_with_sec["_batch"].alias("_batch"),
        lit(run_id).alias("_run_id"),
        current_timestamp().alias("_load_ts")
    )
)

(fact_markethistory.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{gold_db}.fact_markethistory"))

In [0]:
count_company = spark.table(f"{gold_db}.dim_company").count()
count_security = spark.table(f"{gold_db}.dim_security").count()
count_financial = spark.table(f"{gold_db}.financial").count()
count_markethistory = spark.table(f"{gold_db}.fact_markethistory").count()

In [0]:
checks = {
    "gold.dim_company":         (spark.table(f"{gold_db}.dim_company").count(), 5000),
    "gold.dim_security":        (spark.table(f"{gold_db}.dim_security").count(), 8658),
    "gold.financial":           (spark.table(f"{gold_db}.financial").count(), 457025),
     "gold.fact_markethistory":  (spark.table(f"{gold_db}.fact_markethistory").count(), 5285024),
}

print(f"\n{'Table':<30} {'Actual':>12} {'Expected':>12} {'Status'}")
print("-" * 65)
for tbl, (actual, expected) in checks.items():
    status = "PASS" if actual == expected else "FAIL"
    print(f"{tbl:<30} {actual:>12,} {expected:>12,}  {status}")

In [0]:
log_audit_event(spark, carried_run_id, carried_batch, "gold", "dim_company", "OVERWRITE", count_company)
log_gold_recon(spark, carried_run_id, "gold.dim_company", expected_count=5000, actual_count=count_company)

log_audit_event(spark, carried_run_id, carried_batch, "gold", "dim_security", "OVERWRITE", count_security)
log_gold_recon(spark, carried_run_id, "gold.dim_security", expected_count=8658, actual_count=count_security)

log_audit_event(spark, carried_run_id, carried_batch, "gold", "financial", "OVERWRITE", count_financial)
log_gold_recon(spark, carried_run_id, "gold.financial", expected_count=457025, actual_count=count_financial)

log_audit_event(spark, carried_run_id, carried_batch, "gold", "fact_markethistory", "OVERWRITE", count_markethistory)
log_gold_recon(spark, carried_run_id, "gold.fact_markethistory", expected_count=5285024, actual_count=count_markethistory)

sil_cmp = spark.table(f"{silver_db}.company").count()
log_pipeline_recon(spark, carried_run_id, carried_batch, "MARKET", "dim_company", "silver", "gold", sil_cmp, count_company)

sil_sec = spark.table(f"{silver_db}.security").count()
log_pipeline_recon(spark, carried_run_id, carried_batch, "MARKET", "dim_security", "silver", "gold", sil_sec, count_security)

sil_fin = spark.table(f"{silver_db}.financial").count()
log_pipeline_recon(spark, carried_run_id, carried_batch, "MARKET", "financial", "silver", "gold", sil_fin, count_financial)

sil_dm = spark.table(f"{silver_db}.markethistory").count()
log_pipeline_recon(spark, carried_run_id, carried_batch, "MARKET", "fact_markethistory", "silver", "gold", sil_dm, count_markethistory)

# 3. End Pipeline Run
log_domain_run_status(spark, carried_run_id, carried_batch, 'MARKET', 'COMPLETED')
end_pipeline_run(spark, carried_run_id, 'SUCCESS')
log_pipeline_message(spark, carried_run_id, 'INFO', 'gold_market_all', 'Successfully completed processing for gold market tables.')

